# External Test Prep v4
Prep any external T1+lesion dataset (user-specified raw root) into standardized outputs inside this folder.

In [ ]:
from pathlib import Path
import shutil
from src.data_prep.prep_utils import (
    DatasetConfig,
    run_prep,
    combine_standardized,
    run_prep_images_only,
    combine_standardized_images_only,
)

PROJECT_ROOT = Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd()

# --- User inputs ---
HAS_MASKS = True   # True: evaluate with GT masks, False: predict-only workflow
ALREADY_MNI = False
OVERWRITE_PREP = False

# Edit these paths for your external MRI(s)
USER_IMG = PROJECT_ROOT / "your_raw_dataset_here/Images"
USER_MSK = PROJECT_ROOT / "your_raw_dataset_here/Masks"  # ignored when HAS_MASKS=False

OUT_ROOT = PROJECT_ROOT / "data" / "prep_outputs"
DEST = PROJECT_ROOT / "data" / "processed" / "test_input"

if DEST.exists():
    shutil.rmtree(DEST)
DEST.mkdir(parents=True, exist_ok=True)

if HAS_MASKS:
    datasets = [
        DatasetConfig(
            name="TEST",
            images_dir=USER_IMG,
            masks_dir=USER_MSK,
            t1_glob="**/*.nii.gz",
            mask_glob="**/*.nii.gz",
            already_mni=ALREADY_MNI,
            overwrite=OVERWRITE_PREP,
        )
    ]
    outputs = run_prep(datasets, OUT_ROOT, force_overwrite=OVERWRITE_PREP)
    combine_standardized(outputs, DEST)
    print("Prepared test set (images + masks):", DEST)
else:
    out_ds = run_prep_images_only(
        images_dir=USER_IMG,
        out_root=OUT_ROOT,
        name="TEST",
        t1_glob="**/*.nii.gz",
        already_mni=ALREADY_MNI,
        overwrite=OVERWRITE_PREP,
    )
    combine_standardized_images_only([out_ds], DEST)
    print("Prepared test set (images only):", DEST)

print("T1 dir:", DEST / "t1")
print("Mask dir (optional):", DEST / "masks")
print("Manifest:", DEST / "manifest.csv")


## Visualize standardized test set

In [ ]:
from pathlib import Path
from src.data_prep.viewer import show_viewer

PROJECT_ROOT = Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd()
DEST = PROJECT_ROOT / 'data' / 'processed' / 'test_input'
mask_dir = DEST / 'masks'
if mask_dir.exists() and any(mask_dir.glob('*.nii*')):
    show_viewer(DEST)
else:
    t1s = sorted((DEST / 't1').glob('*.nii.gz'))
    print(f"No masks found. Prepared {len(t1s)} image(s) for prediction-only testing.")
    for p in t1s[:10]:
        print(' -', p.name)
